In [0]:
from datetime import datetime

def log_gold(run_id, status, start_time, count, message):

    end_time = datetime.now()

    data = [(run_id, "gold_scd2", status, start_time, end_time, count, message)]

    df = spark.createDataFrame(
        data,
        ["run_id","process","status","start_time","end_time","record_count","message"]
    )

    df.write.mode("append").saveAsTable("fhir_catalog.audit.gold_log")

In [0]:
from pyspark.sql.functions import col

def build_gold_source():

    patient = spark.read.table("fhir_catalog.silver.patient")
    encounter = spark.read.table("fhir_catalog.silver.encounter")
    observation = spark.read.table("fhir_catalog.silver.observation")


    encounter = encounter.withColumn(
        "patient_id",
        col("patient_ref").substr(9, 100)
    )

    observation = observation.withColumn(
        "patient_id",
        col("patient_ref").substr(9, 100)
    )

    encounter_agg = encounter.groupBy("patient_id").count() \
        .withColumnRenamed("count", "total_encounters")

    obs_agg = observation.groupBy("patient_id").count() \
        .withColumnRenamed("count", "total_observations")

    gold_df = patient \
        .join(encounter_agg, "patient_id", "left") \
        .join(obs_agg, "patient_id", "left") \
        .fillna(0)

    return gold_df

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime
import uuid

def gold_scd2():

    run_id = str(uuid.uuid4())
    start_time = datetime.now()

    try:
        source_df = build_gold_source() \
            .withColumn("start_date", current_timestamp()) \
            .withColumn("end_date", lit(None).cast("timestamp")) \
            .withColumn("is_current", lit(True))

        count = source_df.count()

        target_table = "fhir_catalog.gold.patient_analytics_scd"


        if not spark.catalog.tableExists(target_table):
            source_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

            log_gold(run_id, "SUCCESS", start_time, count, "Initial load completed")

            print("Initialized SCD2 table")
            return

        target = DeltaTable.forName(spark, target_table)


        target.alias("t").merge(
            source_df.alias("s"),
            "t.patient_id = s.patient_id AND t.is_current = true"
        ).whenMatchedUpdate(
            condition="""
                t.gender <> s.gender OR
                t.birth_date <> s.birth_date OR
                t.total_encounters <> s.total_encounters OR
                t.total_observations <> s.total_observations
            """,
            set={
                "end_date": current_timestamp(),
                "is_current": lit(False)
            }
        ).execute()


        target.alias("t").merge(
            source_df.alias("s"),
            "t.patient_id = s.patient_id AND t.is_current = true"
        ).whenNotMatchedInsertAll().execute()


        log_gold(run_id, "SUCCESS", start_time, count, "SCD2 merge completed")

        print("SCD2 merge completed successfully")

    except Exception as e:

        log_gold(run_id, "FAILED", start_time, 0, str(e))

        print("Gold FAILED")
        raise

In [0]:
gold_scd2()

In [0]:
%sql
-- select * from fhir_catalog.gold.patient_analytics_scd

In [0]:
%sql
-- select *from fhir_catalog.audit.gold_log